# Session 6: Batch file handling and automated reporting

A spectrometer writes one file per measurement, so a calibration and a set of samples arrive as a folder of files that a person would otherwise open one at a time. This notebook reads such folders with code and writes the result as a table and a short report. Section 1 works through two days of real ferrocyanide spectra: it finds every file, checks the count against the laboratory's own list before reading anything, reads each file with one function in a loop, calibrates on the first day and checks the second day's standards against that calibration, and writes the report. Section 2 does the same for a seeded run of iron in water assigned to you, in which one file will not read. Section 3 asks what to do with that file, and shows that the answer changes a reported concentration. Section 4 is the judgment cell.

Dataset A is real: P. Rimmer, "Supplementary data for 'Measuring Nitroprussside and Ferrocyanide with Quantitative UV-Vis using FlareLab'", Harvard Dataverse, version 1, 2025, [doi:10.7910/DVN/QAZ9B6](https://doi.org/10.7910/DVN/QAZ9B6), CC0 1.0, accessed 2026-09-28, for the preprint by L. Rossmanith, S. K. Platymesi, S. J. Thompson and P. B. Rimmer, ChemRxiv 2025, [doi:10.26434/chemrxiv-2025-d3ltb](https://doi.org/10.26434/chemrxiv-2025-d3ltb). The files in `data/flarelab/` are the folders `Ferrocyanide Standard 240409` and `Ferrocyanide Standard 240410` and the table `Ferrocyanide Standard File Parameters.csv`, unchanged. Dataset B, in `data/runs/`, is simulated; `data/README.md` describes both.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 6 assignment on Canvas by 1:30 pm on Thursday, October 22. Submissions are individual. Every run's files are in the repository, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas, Matplotlib and `linregress`. `Path` is new this session: it is a file or folder location that knows its own parts, such as its name and the folder it sits in, and it can list the files below it. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

## Parameters

Dataset A's folder and parameter table, the two days (written yymmdd, as the files write them), the analytical wavelength for ferrocyanide and the concentration range the calibration uses, in mM, which section 1 explains; then Dataset B's folder, your run, the analytical wavelength for iron, the EPA's secondary drinking-water standard for iron, 0.3 mg/L, which each water in the report is compared with, and the folder the reports are written to. In section 2, `RUN` is the only input a new run changes. Leave `RUN` as `CHANGE-ME` until section 2.

In [ ]:
DATA_DIR = Path("data/flarelab")
TABLE = DATA_DIR / "Ferrocyanide Standard File Parameters.csv"
CAL_DAY = "240409"      # day 1: its standards make the calibration
CHECK_DAY = "240410"    # day 2: its standards are checked against it
WAVELENGTH_NM = 360     # ferrocyanide
CAL_RANGE = (0.5, 10)   # mM; standards used for the calibration, inclusive

RUNS_DIR = Path("data/runs")
RUN = "CHANGE-ME"       # your assigned run, for example "run_07", section 2
IRON_NM = 510           # the iron(II) phenanthroline complex
IRON_LIMIT_MG_L = 0.3   # EPA secondary drinking-water standard for iron
OUTPUT_DIR = Path("output")

## 1. Explore: an experiment is a folder

To know what a batch holds before reading any of it, list the files. `DATA_DIR.glob("*.txt")` returns every path in the folder whose name matches the pattern, where `*` stands for any run of characters; `rglob` does the same in the folder and every folder below it. Both return the paths in whatever order the file system keeps them, which differs between a Windows laptop and a Mac, so `sorted` puts them in name order, the same on every machine.

In [ ]:
top_level = list(DATA_DIR.glob("*.txt"))
paths = sorted(DATA_DIR.rglob("*.txt"))
print(len(top_level), "spectra directly in data/flarelab")
print(len(paths), "spectra in data/flarelab and the folders below it")
for path in paths[:3]:
    print(path.parent.name, "/", path.name)

`glob` found no spectra because they are one level down, in one folder per day. A pattern that matches nothing raises no error: it returns an empty list, and a loop over it does nothing. That is the first reason to count.

The next cell builds the inventory, one row per file, before any file is opened. `path.stem` is the name without `.txt`, and session 5's `split("_")` takes it apart: `parts[1]` is the day, `parts[3]` the distance setting in mm, and `parts[5]` the role, one of `Background` (the dark spectrum), `initial` (the water reference) or `ferrocyanide` (a standard). It then compares the inventory with the laboratory's own list, the parameter table, by count and by name in both directions.

In [ ]:
rows = []   # emptied on every run of the cell, so a second run does not double the table
for path in paths:
    parts = path.stem.split("_")
    rows.append({"file": path.name, "folder": path.parent.name, "day": parts[1],
                 "distance_mm": int(parts[3]), "role": parts[5], "path": path})
inventory = pd.DataFrame(rows)
print(inventory.groupby(["day", "role"]).size())

table = pd.read_csv(TABLE, dtype={"dateID": str})   # the parameter table covers four days; keep the two here
table = table[(table["dateID"] == CAL_DAY) | (table["dateID"] == CHECK_DAY)]
print(len(table), "files in the parameter table for", CAL_DAY, "and", CHECK_DAY, "and", len(inventory), "in the folder")
print("in the table, not in the folder:", sorted(set(table["Filename"]) - set(inventory["file"])))
print("in the folder, not in the table:", sorted(set(inventory["file"]) - set(table["Filename"])))

The counts agree and no name is missing from either side. The next cell reads one file. A Flame export opens with a header of instrument settings and then a marker line, `>>>>>Begin Spectral Data<<<<<`, before two tab-separated columns, wavelength in nm and detector counts. `read_flame` finds the marker the way session 4 found the end of a JCAMP header, as a line in the text, and hands the lines after it to `np.loadtxt`. The plot is day 1 at 50 mm: the dark, the water reference and the 5 mM standard.

In [ ]:
def read_flame(path):
    """Read an Ocean Insight Flame export and return wavelength (nm) and counts as arrays."""
    lines = path.read_text().splitlines()
    start = lines.index(">>>>>Begin Spectral Data<<<<<") + 1
    data = np.loadtxt(lines[start:])
    return data[:, 0], data[:, 1]


day1_50 = inventory[(inventory["day"] == CAL_DAY) & (inventory["distance_mm"] == 50)]
dark_path = day1_50[day1_50["role"] == "Background"]["path"].iloc[0]
reference_path = day1_50[day1_50["role"] == "initial"]["path"].iloc[0]
standard_path = day1_50[day1_50["file"].str.split("_").str[7] == "v8"]["path"].iloc[0]   # v8 is 0.005 M, 5 mM, by the table

fig, ax = plt.subplots()
for path, label, style in [(reference_path, "water reference", "-"), (standard_path, "5 mM standard", "--"),
                           (dark_path, "dark", ":")]:
    wavelength, counts = read_flame(path)
    ax.plot(wavelength, counts, style, color="black", label=label)
ax.axvline(WAVELENGTH_NM, color="gray", linestyle="-.", linewidth=1)
ax.set_xlim(250, 600)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("detector counts")
ax.set_title(f"FlareLab, day {CAL_DAY}, 50 mm")
ax.legend()
fig.tight_layout()
plt.show()

The plot shows three lines from 250 to 600 nm. The water reference (solid) peaks near 530 nm; the 5 mM standard (dashed) follows it above 400 nm and falls toward zero below 350 nm, where ferrocyanide absorbs; the dark (dotted) is flat near the bottom. At 360 nm (the dash-dot line) the reference reads 14,005 counts, the standard 4,860 and the dark 366, the three numbers an absorbance is made from.

`counts_at` returns the counts at one wavelength, interpolated between the two nearest detector pixels with `np.interp`. The loop applies it to every file in the inventory, one dictionary per file, and prints the count read against the count found.

In [ ]:
def counts_at(path, wavelength_nm):
    """Detector counts at one wavelength, interpolated between the two nearest pixels."""
    wavelength, counts = read_flame(path)
    return float(np.interp(wavelength_nm, wavelength, counts))


rows = []
for path in inventory["path"]:
    rows.append({"file": path.name, "counts": counts_at(path, WAVELENGTH_NM)})
read = pd.DataFrame(rows)
print(len(read), "files read of", len(inventory), "in the inventory")

Absorbance is computed from three spectra taken on the same day at the same setting: A = -log10((S - D) / (R - D)). In words, it is minus the base 10 logarithm of the standard's counts over the reference's counts, each less the dark counts. The function is given. The next cell first puts each file's counts and its concentration from the parameter table on its inventory row, merging on the file name; the table calls that column `Filename`, so `left_on` and `right_on` name the column on each side. It then takes out the dark rows and the reference rows, renames their `counts` column, and merges both onto the standards on day and distance setting.

In [ ]:
def absorbance_from_counts(sample, reference, dark):
    """Absorbance from sample, reference and dark counts at the same wavelength."""
    return -np.log10((sample - dark) / (reference - dark))


spectra = inventory.merge(read, on="file")
spectra = spectra.merge(table[["Filename", "concentration (M)"]], left_on="file", right_on="Filename", how="left")

dark = spectra[spectra["role"] == "Background"][["day", "distance_mm", "counts"]].rename(columns={"counts": "dark_counts"})
reference = spectra[spectra["role"] == "initial"][["day", "distance_mm", "counts"]].rename(columns={"counts": "reference_counts"})
standards = spectra[spectra["role"] == "ferrocyanide"]
standards = standards.merge(dark, on=["day", "distance_mm"]).merge(reference, on=["day", "distance_mm"])
standards["concentration_mM"] = 1000 * standards["concentration (M)"]
standards["absorbance_AU"] = absorbance_from_counts(standards["counts"], standards["reference_counts"], standards["dark_counts"])
print(len(standards), "standards:", standards.groupby("day").size().to_dict())

The calibration uses day 1's standards from 0.5 to 10 mM, the range in `CAL_RANGE`, at all three distance settings, and fits a line with `linregress`, whose slope is in AU per mM. The plot shows why the range stops where it does: all eleven day 1 standards, on a logarithmic concentration axis because they span 0.001 to 100 mM. On that axis the straight fitted line looks curved.

In [ ]:
day1 = standards[standards["day"] == CAL_DAY]
in_range = day1[(day1["concentration_mM"] >= CAL_RANGE[0]) & (day1["concentration_mM"] <= CAL_RANGE[1])]
fit = linregress(in_range["concentration_mM"], in_range["absorbance_AU"])
print(f"slope {fit.slope:.5f} AU per mM, intercept {fit.intercept:.5f} AU, R^2 {fit.rvalue ** 2:.5f}, {len(in_range)} standards")

fig, ax = plt.subplots()
for distance, marker in [(50, "o"), (200, "s"), (550, "^")]:
    at = day1[day1["distance_mm"] == distance]
    ax.plot(at["concentration_mM"], at["absorbance_AU"], marker, color="black", fillstyle="none", label=f"{distance} mm")
line_mM = np.linspace(CAL_RANGE[0], CAL_RANGE[1], 50)
ax.plot(line_mM, fit.intercept + fit.slope * line_mM, "-", color="gray", label="fit, 0.5 to 10 mM")
ax.set_xscale("log")
ax.set_xlabel("ferrocyanide (mM)")
ax.set_ylabel(f"absorbance at {WAVELENGTH_NM} nm (AU)")
ax.set_title(f"Day {CAL_DAY} standards")
ax.legend()
fig.tight_layout()
plt.show()

The plot shows the eleven day 240409 standards at three settings (circles, squares, triangles), overlapping at every level. Below 0.5 mM absorbance is within 0.01 AU of zero, from 0.5 to 10 mM it follows the fitted line, and at 50 and 100 mM it stops between 1.78 and 1.82 AU.

The last cell treats day 2's 0.5 to 10 mM standards as unknowns, c = (A - b) / m, i.e., absorbance less intercept, divided by slope. Each gets a recovery, found over nominal in percent. Expect 1, 5 and 10 mM within 85 to 115%, and 0.5 mM far above at every setting, as notes §3 explains. The table and report go to `output/`, overwritten on every run.

In [ ]:
day2 = standards[(standards["day"] == CHECK_DAY) & (standards["concentration_mM"] >= CAL_RANGE[0])
                  & (standards["concentration_mM"] <= CAL_RANGE[1])].copy()
day2["found_mM"] = (day2["absorbance_AU"] - fit.intercept) / fit.slope
day2["recovery_pct"] = 100 * day2["found_mM"] / day2["concentration_mM"]
check = day2[["file", "distance_mm", "concentration_mM", "absorbance_AU", "found_mM", "recovery_pct"]].sort_values(
    ["concentration_mM", "distance_mm"], ignore_index=True)   # ignore_index numbers the sorted rows 0, 1, 2, ...

OUTPUT_DIR.mkdir(exist_ok=True)
check.to_csv(OUTPUT_DIR / "flarelab_summary.csv", index=False)

recovery_by_level = check.groupby("concentration_mM")["recovery_pct"].mean()
lines = [f"# Ferrocyanide check standards, day {CHECK_DAY} against day {CAL_DAY}", ""]
lines.append(f"Files: {len(paths)} found, {len(table)} in the parameter table, {len(read)} read, "
             f"{len(paths) - len(read)} not read.")
lines.append(f"Calibration at {WAVELENGTH_NM} nm on {len(in_range)} day {CAL_DAY} standards, {CAL_RANGE[0]} to {CAL_RANGE[1]} mM: "
             f"slope {fit.slope:.5f} AU per mM, intercept {fit.intercept:.5f} AU, R^2 {fit.rvalue ** 2:.5f}.")
lines += ["", "| Nominal (mM) | Mean recovery (%) |", "|---|---|"]
for nominal, recovery in recovery_by_level.items():
    lines.append(f"| {nominal:g} | {recovery:.1f} |")
report = "\n".join(lines) + "\n"
(OUTPUT_DIR / "flarelab_report.md").write_text(report)
print(report)
check[["distance_mm", "concentration_mM", "absorbance_AU", "found_mM", "recovery_pct"]].round(4)

**Explain.** The notebook compared the folder with the parameter table before it read a single spectrum. Suppose one day 2 standard had been missing from the folder. In two sentences, say what the loop and the recovery table would have shown, and which printed line would have caught the gap.

*Your answer:*

## 2. Apply: your run

Each of you has one run from the list on the board, a folder in `data/runs/` from a teaching laboratory's determination of iron in water: a reagent blank and six iron standards, 0.1 to 4 mg/L, in `standards/`, three waters in triplicate in `samples/`, and the instrument's sequence table, `sequence.csv`, one row per sample position in the order of the sequence. Every spectrum is a two-column CSV, wavelength and absorbance, blanked by the instrument on the reagent blank, `std_0p00`. The report the section writes answers the laboratory's question: is each water above the EPA's secondary drinking-water standard for iron, 0.3 mg/L, the level above which iron stains fixtures and can be tasted? There is nothing to download.

**[TASK 1]** Set `RUN` in the Parameters cell and run that cell again, then run the next cell, which reads the sequence table, lists the spectra one folder down with `glob("*/*.csv")`, builds the inventory and counts it by folder. The last line is the count against the sequence table, with the table's variable left to you. Until `RUN` is set, the cell stops with `FileNotFoundError` for `data/runs/CHANGE-ME/sequence.csv`; until the last line is filled, with `TypeError: object of type 'ellipsis' has no len()`.

In [ ]:
RUN_DIR = RUNS_DIR / RUN
run_sequence = pd.read_csv(RUN_DIR / "sequence.csv", dtype=str)
run_paths = sorted(RUN_DIR.glob("*/*.csv"))

run_listing = []
for path in run_paths:
    parts = path.stem.split("_")
    run_listing.append({"file": f"{path.parent.name}/{path.name}", "folder": path.parent.name,
                        "name": parts[0], "tag": parts[1]})
run_inventory = pd.DataFrame(run_listing)
print(run_inventory.groupby("folder").size())
print(len(run_inventory), "spectra found,", len(...), "in the sequence table")   # [TASK 1] the table read from sequence.csv

The reader for these files is shorter than `read_flame`, because they are plain CSV: `absorbance_at` reads one file with `read_csv` and returns the absorbance at one wavelength with `np.interp`.

In [ ]:
def absorbance_at(path, wavelength_nm):
    """Absorbance at one wavelength from a two-column CSV spectrum."""
    spectrum = pd.read_csv(path)
    return float(np.interp(wavelength_nm, spectrum["wavelength_nm"], spectrum["absorbance_AU"]))

One file in every run does not read, and a plain loop stops at it with an error, so the loop in TASK 2 is wrapped in `try` and `except`. Python runs the `try` block; if a line in it raises an error, the rest of the block is skipped and the `except` block runs instead, with the error available as `err`, and then the loop goes on to the next file. What the `except` block does with that file is the decision.

**Ask the assistant** before the next task, in Claude Code, and give the assistant a check to run, today's AI practice: paste the loop below and ask `This loop reads every file in my run. What should the except branch do so the loop keeps going when a file will not read? Then run your version of the loop in my notebook and print how many files it read and how many it logged, next to the number of rows in run_sequence.` To run code, the assistant adds a cell at the end of the notebook and asks you to click **Execute** or **Cancel**. If no **Execute** button appears, click **+ Code** in the notebook toolbar, paste the assistant's code into the new cell and run it yourself. The three counts in that cell's output are the check; the sentence in the chat that reports them is a claim. Read them against TASK 3, which counts what your own loop kept, then delete the cell the assistant or you added, so that the notebook still ends with the software environment cell. What it suggested and what you did with it is one candidate for part (b) of the judgment cell.

**[TASK 2]** Replace the `raise` line in the `except` block with what the run keeps about a file that will not read, or with nothing. If you keep a record, append it to `run_failures`; the `failed` table and the report in TASK 4 read a record as a dictionary with the keys `"file"` and `"error"`. Until the line is replaced, `raise` passes the error on and the cell stops with `ParserError: Error tokenizing data. C error: Expected 1 fields in line 7, saw 2`, the same stop as a loop with no `try`.

In [ ]:
run_rows = []       # one dict per file that read, emptied on every run of the cell
run_failures = []   # one dict per file that did not
for path in run_paths:
    name = f"{path.parent.name}/{path.name}"   # the same form as the sequence table's file column
    try:
        run_rows.append({"file": name, "absorbance_AU": absorbance_at(path, IRON_NM)})
    except Exception as err:
        raise   # [TASK 2] replace this line: what the run keeps about a file that will not read
failed = pd.DataFrame(run_failures, columns=["file", "error"])
print(len(run_rows), "files read,", len(failed), "logged")
failed

**[TASK 3]** The reconciliation: every file in the sequence table is either a row that read or a file the loop logged, so the two add up to the table's count. Fill the stub with the number of files the loop logged; until it is filled, the cell stops with `TypeError: unsupported operand type(s) for +: 'int' and 'ellipsis'`. The last number printed is the count of files neither read nor logged.

In [ ]:
accounted = len(run_rows) + ...   # [TASK 3] the number of files the loop logged
print(f"{len(run_sequence)} in the sequence table, {len(run_inventory)} found, {len(run_rows)} read, "
      f"{len(run_failures)} logged, {len(run_sequence) - accounted} unaccounted for")

The next cell is given. It merges the rows that read with the sequence table, fits the calibration on the seven standards, finds each sample replicate's concentration by c = (A - b) / m, and summarizes each water as a mean, a standard deviation and the number of replicates, n. The standard deviation is the sample standard deviation, pandas' default, which divides by n - 1.

In [ ]:
results = pd.DataFrame(run_rows).merge(run_sequence, on="file", how="left")
run_standards = results[results["kind"] == "standard"]
run_fit = linregress(run_standards["level_mg_L"].astype(float), run_standards["absorbance_AU"])
replicates = results[results["kind"] == "sample"].copy()
replicates["iron_mg_L"] = (replicates["absorbance_AU"] - run_fit.intercept) / run_fit.slope


def summarize(replicates):
    """One row per water: mean, sample standard deviation and n of its replicates."""
    by_water = replicates.groupby("sample")["iron_mg_L"]
    return pd.DataFrame({"water": by_water.mean().index, "iron_mg_L": by_water.mean().values,
                         "sd_mg_L": by_water.std().values, "n": by_water.size().values})


print(f"slope {run_fit.slope:.4f} AU per mg/L, intercept {run_fit.intercept:.4f} AU, R^2 {run_fit.rvalue ** 2:.5f}")
summary_A = summarize(replicates)
summary_A.round(4)

**Explain.** Before you compare: in two sentences, give the file in your run that did not read, its water and replicate, and what its first line says. Open it from the Explorer panel in VS Code, under `data/runs/`, your run, `samples/`. Then compare with a neighbor: set your two files, waters, replicates and first lines side by side. Add one sentence on where your two answers differ and what in your `sequence.csv` explains it.

*Your answer:*

## 3. Analyze: the file that did not read

The file that did not read is found here from the inventory, as the one name with no row, so this section works whatever TASK 2 kept. The cell prints its first eight lines; read them as well as counting them. It opens with a header of six lines that the other files lack, and a plain `read_csv` takes the first line as the column names and then meets rows with two fields where it expected one. `skiprows=6` starts the read below the header, and that one file then reads.

In [ ]:
unread = sorted(set(run_inventory["file"]) - set(results["file"]))
unread_name = unread[0]
unread_path = RUN_DIR / unread_name
print(unread_name)
print("\n".join(unread_path.read_text().splitlines()[:8]))

unread_spectrum = pd.read_csv(unread_path, skiprows=6)
unread_absorbance = float(np.interp(IRON_NM, unread_spectrum["wavelength_nm"], unread_spectrum["absorbance_AU"]))
unread_row = run_sequence[run_sequence["file"] == unread_name]
affected = unread_row["sample"].iloc[0]
print(f"absorbance at {IRON_NM} nm {unread_absorbance:.4f} AU; the sequence table lists it as {affected}, "
      f"replicate {unread_row['replicate'].iloc[0]}, on {unread_row['instrument'].iloc[0]}")

Two treatments of that file are defensible, and the report has to use one. Treatment A excludes it and reports the water from the other two replicates, n = 2. Treatment B reads it with `skiprows` and includes it, n = 3. The cell builds the summary under B, a copy of the summary under A with the affected water's row recomputed from three values by `np.mean` and `np.std`, whose `ddof=1` makes it the same n - 1 standard deviation as pandas', and prints that water under both, with the file's own value and whether each is above 0.3 mg/L, so the two can be compared number by number.

In [ ]:
unread_mg_L = (unread_absorbance - run_fit.intercept) / run_fit.slope
values_A = replicates[replicates["sample"] == affected]["iron_mg_L"].tolist()
values_B = values_A + [unread_mg_L]
summary_B = summary_A.copy()
is_affected = summary_B["water"] == affected
summary_B.loc[is_affected, "iron_mg_L"] = np.mean(values_B)
summary_B.loc[is_affected, "sd_mg_L"] = np.std(values_B, ddof=1)
summary_B.loc[is_affected, "n"] = len(values_B)

print(f"{affected}, the replicates that read: {np.round(values_A, 4)} mg/L")
print(f"{affected}, the file that did not read: {unread_mg_L:.4f} mg/L")
for label, summary in [("A, excluded", summary_A), ("B, included", summary_B)]:
    row = summary[summary["water"] == affected]
    mean = row["iron_mg_L"].iloc[0]
    print(f"treatment {label}: {mean:.4f} mg/L, SD {row['sd_mg_L'].iloc[0]:.4f} mg/L, n = {row['n'].iloc[0]}, "
          f"{'above' if mean > IRON_LIMIT_MG_L else 'not above'} {IRON_LIMIT_MG_L} mg/L")

**Explain.** Give the affected water's concentration, standard deviation and n under each treatment, say by how much the standard deviation changes when the file is included, and say whether either treatment puts the water on the other side of 0.3 mg/L. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

**[TASK 4]** Choose the treatment your report uses, `"A"` or `"B"`, and run the cell. It writes `<RUN>_summary.csv` and `<RUN>_report.md` to `output/`, overwritten on every run, and prints the report, which opens with the file counts from TASK 3. Until `TREATMENT` is set, the cell stops with `KeyError: Ellipsis`.

In [ ]:
TREATMENT = ...   # [TASK 4] "A" or "B", the treatment of the UV-2 file your report uses

report_summary = {"A": summary_A, "B": summary_B}[TREATMENT]
OUTPUT_DIR.mkdir(exist_ok=True)
report_summary.to_csv(OUTPUT_DIR / f"{RUN}_summary.csv", index=False)

lines = [f"# Iron in water, {RUN}", ""]
lines.append(f"Files: {len(run_sequence)} in the sequence table, {len(run_inventory)} found, "
             f"{len(run_rows)} read, {len(run_failures)} logged as not read.")
for record in run_failures:
    lines.append(f"Not read: {record['file']} ({record['error']})")
lines.append(f"Calibration at {IRON_NM} nm on {len(run_standards)} standards: slope {run_fit.slope:.4f} AU per mg/L, "
             f"intercept {run_fit.intercept:.4f} AU, R^2 {run_fit.rvalue ** 2:.5f}.")
lines.append(f"Treatment {TREATMENT} of {unread_name}: " + ("excluded." if TREATMENT == "A" else "read with skiprows and included."))
lines += ["", f"| Water | Iron (mg/L) | SD (mg/L) | n | Above {IRON_LIMIT_MG_L} mg/L |", "|---|---|---|---|---|"]
for water, mean, sd, n in zip(report_summary["water"], report_summary["iron_mg_L"],
                              report_summary["sd_mg_L"], report_summary["n"]):
    lines.append(f"| {water} | {mean:.3f} | {sd:.3f} | {n} | {'yes' if mean > IRON_LIMIT_MG_L else 'no'} |")
report = "\n".join(lines) + "\n"
(OUTPUT_DIR / f"{RUN}_report.md").write_text(report)
print(report)

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): say which treatment went into your report and why, with the affected water's concentration, standard deviation and n under both and whether either changes its comparison with 0.3 mg/L, and say what your summary would have shown for that water, and what your report would have said about the file counts, had the `except` branch kept nothing. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your run. The prompt before TASK 2 is the natural source; the reason has to cite a count or a value from your run, not a general rule. This session's AI practice, named in the notes, is *give the assistant a check to run*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")